# 10 · DLinear (linear)

Zeng et al. (2023), *Are Transformers Effective for Time Series Forecasting?* Decompõe a janela em **tendência** (média móvel de 25 passos) e **resto sazonal**, e aplica uma camada linear L → H em cada parte.

É o modelo linear de referência: se arquiteturas mais complexas não o superam, a complexidade extra não se justifica. Hiperparâmetros do artigo original (média móvel de 25 passos).

In [1]:
# Configuração: igual em todos os notebooks
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJETO = '/content/drive/MyDrive/RN-Projeto'
except ImportError:          # fora do Colab (ex.: rodando no computador)
    PROJETO = os.path.abspath('.')
sys.path.insert(0, os.path.join(PROJETO, 'src'))
print('Pasta do projeto:', PROJETO)

import importlib
import rn_utils as ru
importlib.reload(ru)        # pega a versão mais nova se o 02 foi rodado de novo
ru.configurar(PROJETO)

Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/RN-Projeto
Projeto: /content/drive/MyDrive/RN-Projeto | dispositivo: cpu


## Modelo

`criar(L, H)` devolve um `nn.Module` que recebe `[lote, L, 1]` e devolve `[lote, H, 1]`. A normalização por janela (RevIN) é aplicada por fora, pelo `ru.ComRevIN`, igual para todos os modelos.

In [2]:
import torch
import torch.nn as nn

class MediaMovel(nn.Module):
    """Média móvel com repetição das bordas (mantém o tamanho L)."""
    def __init__(self, k):
        super().__init__()
        self.k = k
        self.pool = nn.AvgPool1d(k, stride=1)

    def forward(self, x):                                   # x: [B, L, C]
        ini = x[:, :1].repeat(1, (self.k - 1) // 2, 1)
        fim = x[:, -1:].repeat(1, self.k // 2, 1)
        x = torch.cat([ini, x, fim], dim=1)
        return self.pool(x.transpose(1, 2)).transpose(1, 2)

class DLinear(nn.Module):
    def __init__(self, L, H, k=25):
        super().__init__()
        self.decomp = MediaMovel(k)
        self.linear_sazonal = nn.Linear(L, H)
        self.linear_tendencia = nn.Linear(L, H)
        # inicialização do artigo: cada saída começa como a média da janela
        for lin in (self.linear_sazonal, self.linear_tendencia):
            nn.init.constant_(lin.weight, 1.0 / L)
            nn.init.zeros_(lin.bias)

    def forward(self, x):                                   # [B, L, 1] → [B, H, 1]
        tendencia = self.decomp(x)
        sazonal = x - tendencia
        y = self.linear_sazonal(sazonal.transpose(1, 2)) + self.linear_tendencia(tendencia.transpose(1, 2))
        return y.transpose(1, 2)

def criar(L, H):
    return DLinear(L, H)

In [3]:
# Verificação do formato de entrada e saída
import torch
for L in (96, 720):
    m = criar(L, ru.HORIZONTE)
    saida = m(torch.randn(4, L, 1))
    assert saida.shape == (4, ru.HORIZONTE, 1), saida.shape
    print(f'L={L}: saída {tuple(saida.shape)} ✓ | {sum(p.numel() for p in m.parameters()):,} parâmetros')

L=96: saída (4, 96, 1) ✓ | 18,624 parâmetros
L=720: saída (4, 96, 1) ✓ | 138,432 parâmetros


## Treino e avaliação

O learning rate é escolhido pela validação (L = 336, seed 0) entre {1e-3, 3e-4, 1e-4}. Depois o modelo é treinado em todas as janelas e seeds das duas bases e avaliado no teste, com a janela original e com as quatro janelas embaralhadas. As execuções já salvas em `resultados/` são apenas recarregadas e listadas.

In [4]:
ru.rodar_grade('DLinear', criar)

lr escolhido para DLinear em ETTh1: 0.001  (val: {'0.001': 0.0908, '0.0003': 0.091, '0.0001': 0.091})
  · ETTh1_DLinear_L96_H96_lr0.001_s0 (salvo) | val 0.0989 | teste MSE 0.0553 MAE 0.1781 | embaralhar_tudo ×1.29 | 8 épocas, 3s (cpu)
  · ETTh1_DLinear_L96_H96_lr0.001_s1 (salvo) | val 0.0991 | teste MSE 0.0554 MAE 0.1782 | embaralhar_tudo ×1.32 | 13 épocas, 7s (cpu)
  · ETTh1_DLinear_L96_H96_lr0.001_s2 (salvo) | val 0.0991 | teste MSE 0.0549 MAE 0.1775 | embaralhar_tudo ×1.34 | 17 épocas, 7s (cpu)
  · ETTh1_DLinear_L192_H96_lr0.001_s0 (salvo) | val 0.0919 | teste MSE 0.0550 MAE 0.1782 | embaralhar_tudo ×1.44 | 13 épocas, 9s (cpu)
  · ETTh1_DLinear_L192_H96_lr0.001_s1 (salvo) | val 0.0918 | teste MSE 0.0551 MAE 0.1783 | embaralhar_tudo ×1.42 | 10 épocas, 7s (cpu)
  · ETTh1_DLinear_L192_H96_lr0.001_s2 (salvo) | val 0.0923 | teste MSE 0.0555 MAE 0.1792 | embaralhar_tudo ×1.37 | 7 épocas, 4s (cpu)
  · ETTh1_DLinear_L336_H96_lr0.001_s0 (salvo) | val 0.0908 | teste MSE 0.0543 MAE 0.1788 | em

## Resumo (média das 3 seeds)

`×tudo`, `×antigo`, `×recente` e `×blocos` = MSE com a janela embaralhada / MSE original. ×1,00 significa que o modelo ignora aquela ordem.

In [5]:
df = ru.carregar_resultados()
r = df[df.modelo == 'DLinear']
resumo = r.groupby(['dataset', 'L']).agg(**{
    'MSE val': ('mse_val', 'mean'), 'MSE teste': ('original_mse', 'mean'), 'dp': ('original_mse', 'std'),
    'MAE teste': ('original_mae', 'mean'), 'MSE sem suspeitos': ('original_mse_ss', 'mean'),
    **{'×' + p.replace('embaralhar_', ''): (f'{p}_razao', 'mean') for p in ru.PERTURBACOES},
    'épocas': ('epocas_rodadas', 'mean'), 'tempo (s)': ('tempo_treino_s', 'mean')})
resumo.round(4)

MSE val  MSE teste      dp  MAE teste  MSE sem suspeitos   ×tudo  \
dataset L                                                                       
ETTh1   96    0.0990     0.0552  0.0003     0.1779             0.0546  1.3166   
        192   0.0920     0.0552  0.0003     0.1786             0.0563  1.4098   
        336   0.0905     0.0548  0.0008     0.1792             0.0571  1.4958   
        512   0.0903     0.0568  0.0022     0.1849             0.0592  1.3467   
        720   0.0933     0.0600  0.0034     0.1895             0.0624  1.3210   
Weather 96    0.5831     0.4860  0.0026     0.4829             0.4860  1.3178   
        336   0.5277     0.4041  0.0016     0.4429             0.4041  1.4654   
        720   0.5142     0.4024  0.0016     0.4448             0.4024  1.6983   

             ×antigo  ×recente  ×blocos   épocas  tempo (s)  
dataset L                                                    
ETTh1   96    1.0362    1.1135   1.3940  12.6667     5.4164  
        192   1.0368    1.1814   1.5221  10.0000     6.6935  
        336   1.0101    1.2371   1.6172  10.6667    10.5726  
        512   0.9069    1.2572   1.5645   9.0000    12.3360  
        720   0.8952    1.2711   1.5396   7.6667    13.5217  
Weather 96    1.0111    1.1163   1.3339  27.6667    50.7963  
        336   1.0780    1.2304   1.6186  19.3333    75.6003  
        720   1.0495    1.2997   1.8690  18.3333   128.2172